# ECG Biomarker Encoder Analysis & Visualization

This notebook loads the evaluation metrics and latent space representations from the three trained models (Attention MLP Autoencoder, $eta$-VAE, and FT-Transformer Autoencoder) and plots comparisons of their reconstruction quality, embedding spaces, and downstream clinical representation utility.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

metrics_path = "../biomarker_encoder/outputs/model_comparison_metrics.csv"
viz_data_path = "../biomarker_encoder/outputs/visualization_data.json"
features_path = "../previous_version/biomarker_encoder/ecg_features.csv"

if not os.path.exists(metrics_path):
    # Fallback to local path if run from project root
    metrics_path = "biomarker_encoder/outputs/model_comparison_metrics.csv"
    viz_data_path = "biomarker_encoder/outputs/visualization_data.json"
    features_path = "previous_version/biomarker_encoder/ecg_features.csv"

### 1. Load Metrics Table

In [ ]:
df_metrics = pd.read_csv(metrics_path)
df_metrics

### 2. Plot Reconstruction Error Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_metrics, x="model_type", y="MSE", ax=axes[0], palette="viridis")
axes[0].set_title("Reconstruction Mean Squared Error (MSE) - Lower is Better", fontsize=12, fontweight="bold")
axes[0].set_ylabel("MSE")
axes[0].set_xlabel("Model Type")

sns.barplot(data=df_metrics, x="model_type", y="MAE", ax=axes[1], palette="viridis")
axes[1].set_title("Reconstruction Mean Absolute Error (MAE) - Lower is Better", fontsize=12, fontweight="bold")
axes[1].set_ylabel("MAE")
axes[1].set_xlabel("Model Type")

plt.tight_layout()
plt.show()

### 3. Plot Downstream Task Utility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_metrics, x="model_type", y="Downstream_F1_Score", ax=axes[0], palette="magma")
axes[0].set_title("Downstream Task Macro F1 Score - Higher is Better", fontsize=12, fontweight="bold")
axes[0].set_ylabel("F1 Score")
axes[0].set_xlabel("Model Type")

sns.barplot(data=df_metrics, x="model_type", y="Downstream_ROC_AUC", ax=axes[1], palette="magma")
axes[1].set_title("Downstream Task Macro ROC-AUC - Higher is Better", fontsize=12, fontweight="bold")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_xlabel("Model Type")

plt.tight_layout()
plt.show()

### 4. Visualize Latent Space Clusters (t-SNE)

In [ ]:
with open(viz_data_path, "r") as f:
    viz_data = json.load(f)
    
features_df = pd.read_csv(features_path)
# Determine labels for color coding (dominant diagnosis superclass)
label_cols = ["NORM", "MI", "STTC", "CD", "HYP"]
labels = np.zeros(len(features_df))
for idx, row in features_df.iterrows():
    for c_idx, col in enumerate(label_cols):
        if col in str(row["diagnostic_superclasses"]):
            labels[idx] = c_idx
            break

# Split labels to align with test set (taking last 15%)
test_size = int(len(features_df) * 0.15)
test_labels = labels[-test_size:]
label_names = [label_cols[int(i)] for i in test_labels]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
model_types = ["attention_mlp", "beta_vae", "ft_transformer"]

for idx, model_type in enumerate(model_types):
    data = viz_data[model_type]
    # Align labels with the test length
    cur_labels = label_names[:len(data["tsne_x"])]
    
    sns.scatterplot(
        x=data["tsne_x"],
        y=data["tsne_y"],
        hue=cur_labels,
        palette="Set2",
        ax=axes[idx],
        s=70,
        alpha=0.8
    )
    axes[idx].set_title(f"{model_type.upper()} Embedding Space (t-SNE)", fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("t-SNE Dimension 1")
    axes[idx].set_ylabel("t-SNE Dimension 2")
    axes[idx].legend(title="Diagnostic Superclass", loc="best")

plt.tight_layout()
plt.show()

### 5. Reconstruction Comparison Example

In [ ]:
sample_idx = 0
model_type = df_metrics.loc[df_metrics["MSE"].idxmin(), "model_type"] # best model
data = viz_data[model_type]

original_vector = np.array(data["inputs"][sample_idx])
reconstructed_vector = np.array(data["reconstructed"][sample_idx])

indices = np.arange(len(original_vector))

plt.figure(figsize=(15, 6))
plt.bar(indices - 0.2, original_vector, width=0.4, label="Original Normalized Biomarkers", color="#1f77b4")
plt.bar(indices + 0.2, reconstructed_vector, width=0.4, label=f"Reconstructed ({model_type.upper()})", color="#ff7f0e")
plt.title(f"Reconstruction Comparison for Sample 1 (using {model_type.upper()})", fontsize=14, fontweight="bold")
plt.xlabel("Biomarker Feature Index")
plt.ylabel("Normalized Feature Value")
plt.xticks(indices)
plt.legend()
plt.tight_layout()
plt.show()